# Modified Bass Model Homework 2

Ноутбук реализует модифицированную модель Басса для двух компаний, оценивает сходимость,
генерирует датасет для ML и решает обратную задачу.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modified_bass_model import ModifiedBassModel, generate_dataset

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:0.4f}')

def build_features(df, degree):
    pA = df['pA'].to_numpy()
    pB = df['pB'].to_numpy()
    columns = [np.ones(len(df)), pA, pB]
    if degree >= 2:
        columns.extend([pA**2, pA * pB, pB**2])
    if degree >= 3:
        columns.extend([pA**3, (pA**2) * pB, pA * (pB**2), pB**3])
    return np.column_stack(columns)

def fit_polynomial_regression(train_df, target, degree):
    X = build_features(train_df, degree)
    y = target.to_numpy()
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    return coef

def predict_polynomial_regression(df, coef, degree):
    X = build_features(df, degree)
    pred = X @ coef
    return np.clip(pred, 0.0, 1.0)

def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def r2(y_true, y_pred):
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0


## 1. Базовая симуляция

Запускаем модель на 200 шагов и смотрим на динамику рынка.


In [ ]:
model = ModifiedBassModel(
    N=10_000,
    pA=0.012,
    qA=0.32,
    pB=0.010,
    qB=0.28,
    disappointment=0.025,
    tolerance=0.08,
    aggression=0.12,
)
history = model.run(steps=200)
history.tail()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
history.plot(x='step', y=['potential', 'A_total', 'B_total'], ax=axes[0])
axes[0].set_title('Population dynamics')
axes[0].set_ylabel('Customers')

history.plot(x='step', y=['share_A', 'share_B'], ax=axes[1])
axes[1].set_title('Market shares')
axes[1].set_ylabel('Share')
plt.tight_layout()


## 2. Оценка сходимости

Сходимость считаем достигнутой, если максимальное отклонение `share_A` на последних 20 шагах меньше `1e-3`.


In [ ]:
summary = model.convergence_summary(steps=200, window=20, tolerance_value=1e-3)
summary


In [ ]:
tail_window = history.tail(40).copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

history.plot(x='step', y=['share_A', 'share_B'], ax=axes[0])
axes[0].axvspan(history['step'].iloc[-20], history['step'].iloc[-1], color='orange', alpha=0.15)
axes[0].set_title('Shares over full horizon')
axes[0].set_ylabel('Share')

tail_window.plot(x='step', y=['share_A', 'share_B'], ax=axes[1], marker='o')
axes[1].set_title('Tail window for convergence check')
axes[1].set_ylabel('Share')
plt.tight_layout()


In [ ]:
tail_window[['step', 'share_A', 'share_B']].tail(20)


## 3. Варьирование параметров

Варьируем `pA` и `pB`, остальные параметры фиксируем. Получаем датасет из предельных долей рынка.


In [ ]:
pA_values = np.linspace(0.005, 0.050, 16)
pB_values = np.linspace(0.005, 0.050, 16)

dataset = generate_dataset(
    pA_values,
    pB_values,
    qA=0.30,
    qB=0.30,
    disappointment=0.02,
    tolerance=0.05,
    aggression=0.10,
    steps=200,
)
dataset.head()


In [ ]:
dataset[['share_A', 'max_share_deviation']].describe()


In [ ]:
pivot = dataset.pivot(index='pA', columns='pB', values='share_A')

plt.figure(figsize=(8, 6))
plt.imshow(
    pivot.values,
    origin='lower',
    aspect='auto',
    extent=[pivot.columns.min(), pivot.columns.max(), pivot.index.min(), pivot.index.max()],
    cmap='viridis'
)
plt.colorbar(label='Equilibrium share_A')
plt.xlabel('pB')
plt.ylabel('pA')
plt.title('Equilibrium share of company A')
plt.show()


## 4. ML-модель

Сравниваем три регрессионные модели: линейную, квадратичную и кубическую. Все они обучаются через `numpy.linalg.lstsq`.


In [ ]:
test_mask = ((dataset.index % 4) == 0)
train_df = dataset.loc[~test_mask, ['pA', 'pB']].reset_index(drop=True)
test_df = dataset.loc[test_mask, ['pA', 'pB']].reset_index(drop=True)
y_train = dataset.loc[~test_mask, 'share_A'].reset_index(drop=True)
y_test = dataset.loc[test_mask, 'share_A'].reset_index(drop=True)

degrees = {
    'linear': 1,
    'quadratic': 2,
    'cubic': 3,
}

results = []
models = {}
for name, degree in degrees.items():
    coef = fit_polynomial_regression(train_df, y_train, degree)
    pred = predict_polynomial_regression(test_df, coef, degree)
    results.append({
        'model': name,
        'degree': degree,
        'mae': mae(y_test.to_numpy(), pred),
        'r2': r2(y_test.to_numpy(), pred),
    })
    models[name] = coef

results_df = pd.DataFrame(results).sort_values('mae').reset_index(drop=True)
results_df


In [ ]:
best_model_name = results_df.loc[0, 'model']
best_degree = int(results_df.loc[0, 'degree'])
best_coef = models[best_model_name]
y_pred = predict_polynomial_regression(test_df, best_coef, best_degree)

print('Best model:', best_model_name)
print('MAE:', mae(y_test.to_numpy(), y_pred))
print('R2:', r2(y_test.to_numpy(), y_pred))

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.8)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='black', linestyle='--')
plt.xlabel('True share_A')
plt.ylabel('Predicted share_A')
plt.title(f'Prediction quality: {best_model_name}')
plt.show()


## 5. Обратная задача и экономика

Выбираем целевую долю рынка компании A и ищем область параметров, где прогноз отличается не больше чем на 7%.


In [ ]:
full_features = dataset[['pA', 'pB']].reset_index(drop=True)
full_prediction = predict_polynomial_regression(full_features, best_coef, best_degree)
target_share = 0.60
mask = np.abs(full_prediction - target_share) <= 0.07
inverse_region = dataset.loc[mask].copy()
inverse_region['predicted_share_A'] = full_prediction[mask]

print('Target share_A:', target_share)
print('Pairs in inverse region:', len(inverse_region))
inverse_region.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(dataset['pB'], dataset['pA'], c='lightgray', s=35, label='all grid points')
scatter = axes[0].scatter(
    inverse_region['pB'],
    inverse_region['pA'],
    c=inverse_region['predicted_share_A'],
    cmap='plasma',
    s=55,
    label='inverse region'
)
axes[0].set_xlabel('pB')
axes[0].set_ylabel('pA')
axes[0].set_title('Inverse region within 7%')
axes[0].legend()
fig.colorbar(scatter, ax=axes[0], label='Predicted share_A')

history.plot(x='step', y=['cost_A', 'cost_B'], ax=axes[1])
axes[1].set_title('Company costs over time')
axes[1].set_ylabel('Cost')
plt.tight_layout()
